In [ ]:
from torch_geometric.datasets import TUDataset
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool

dataset = TUDataset(root='data/TUDataset', name='ENZYMES')
print(f'Dataset: {dataset}:')
print('====================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of nodes: {dataset.data.num_nodes}')
print(f'Number of features: {dataset.num_node_features}')
print(f'Number of classes: {dataset.num_classes}')

Dataset: ENZYMES(600):
Number of graphs: 600
Number of nodes: 19580
Number of features: 3
Number of classes: 6


/var/folders/kl/360yp2fx3wl5t7zzf039z1mw0000gn/T/ipykernel_32159/2502150174.py:10: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  print(f'Number of nodes: {dataset.data.num_nodes}')


In [19]:
from torch_geometric.loader import DataLoader

train_dataset = dataset[:int(0.8 * len(dataset))]
val_dataset = dataset[int(0.8 * len(dataset)):int(0.9 * len(dataset))]
test_dataset = dataset[int(0.9 * len(dataset)):]

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [20]:
print(f'Number of training graphs: {len(train_dataset)}')
print(f'Number of validation graphs: {len(val_dataset)}')
print(f'Number of test graphs: {len(test_dataset)}')

Number of training graphs: 480
Number of validation graphs: 60
Number of test graphs: 60


In [ ]:
class GIN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super(GIN, self).__init__()

        self.conv1 = GINConv(
            nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
        )

        self.conv2 = GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
        )

        self.conv3 = GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
        )
    
        self.lin1 = nn.Linear(hidden_dim * 3, hidden_dim * 3)
        self.lin2 = nn.Linear(hidden_dim * 3, num_classes)

    def forward(self, x, edge_index, batch):
        x1 = self.conv1(x, edge_index)
        x2 = self.conv2(x1, edge_index)
        x3 = self.conv3(x2, edge_index)

        x1 = global_add_pool(x1, batch)
        x2 = global_add_pool(x2, batch)
        x3 = global_add_pool(x3, batch)

        x = torch.cat([x1, x2, x3], dim=1)
        
        x = self.lin1(x)
        x = nn.ReLU()(x)
        x = nn.Dropout(p=0.5)(x)
        x = self.lin2(x)

        return x
